# Demo 1: Estimate Perceived Safety Scores considering different models and compare

## Import general packages

In [ ]:
# import pandas as pd
import copy
import os
import geopandas as gpd
# import matplotlib

## Import PsafeChoices package and its functions

In [ ]:
from Psafechoices.calc import processRowEst
from Psafechoices.mapAnalysis import plotPsafeLev, PsafeHeatmaps
from Psafechoices.mainFuns import modelPsafe_import, linksPsafe_import, score_diff

## Step 1: Import the links for which pereived safety score will be estimated

In [ ]:
network_city = "Munich"
version = "v12"
# GeoPackage URL from Zenodo
gpkg_url = ("https://zenodo.org/records/20537554/files/" f"baseNetwork{network_city}Links_{version}.gpkg?download=1")

links = gpd.read_file(gpkg_url)

## Step 2.1: Import the perceived safety models

In [ ]:
models_dir = ""
models_version = "_v1.1" # this one is with safety perceptions of Athens residents

cf1 = modelPsafe_import(os.path.join(models_dir, "models" + models_version, 
                                    "psafe", "psafe_models.csv"))
cf1

In [ ]:
models_version = "_v1.2" # this one is with safety perceptions of Munich residents

cf2 = modelPsafe_import(os.path.join(models_dir, "models" + models_version, 
                                    "psafe", "psafe_models.csv"))
cf2

## Step 3.1: Estimate perceived safety scores based on different safety perceptions

In [ ]:
modes = ['car', 'ebike', 'escoot', 'walk']
links_1 = links.copy()
for m in modes:
    latent_vars = []
    safety_levels = []
    
    for index, row in links.iterrows():        
        latent_vars.append(processRowEst(index, row, modes, cf1)[f'LatPsafe{m}'])
        safety_levels.append(processRowEst(index, row, modes, cf1)[f'LevPsafe{m}'])

    links_1[f'LatPsafe{m}'] = latent_vars
    links_1[f'LevPsafe{m}'] = safety_levels

# If you want to plot the scores for each
for m in modes: plotPsafeLev(links_1, m, city = network_city)
# If you want to create a heatmap, showing the density of safe links per mode
for m in modes: PsafeHeatmaps(copy.deepcopy(links_1), m, city = network_city)

## Step 3.2: Estimate perceived safety scores based on different safety perceptions

In [ ]:
links_2 = links.copy()
for m in modes:
    latent_vars = []
    safety_levels = []
    
    for index, row in links.iterrows():        
        latent_vars.append(processRowEst(index, row, modes, cf2)[f'LatPsafe{m}'])
        safety_levels.append(processRowEst(index, row, modes, cf2)[f'LevPsafe{m}'])

    links_2[f'LatPsafe{m}'] = latent_vars
    links_2[f'LevPsafe{m}'] = safety_levels

# for m in modes: plotPsafeLev(links_2, m, city = network_city)
# for m in modes: PsafeHeatmaps(copy.deepcopy(links_2), m, city = network_city)

## Step 4: Estimate significant differences

In [ ]:
r = 50 # draws, which 50 different travelers with different "tastes" regarding the influence of road infrastructure type
df = score_diff(links_1, links_2, cf1, cf2, runs = r)